# Bài 5 · Pure Monte Carlo Search

`playout()` thử một nước rồi để hai bên đi ngẫu nhiên đến khi kết thúc. `playouts()` lặp lại việc đó để ước lượng utility trung bình theo góc nhìn tác tử. `pmcs()` phân bổ ngân sách **tổng cộng** N gần như đều giữa các nước đi. Phần dư được gán ngẫu nhiên sao cho chênh lệch tối đa một lượt. Tác tử chọn nước có trung bình cao nhất, ưu tiên ô có chỉ số nhỏ hơn khi bằng điểm.

In [1]:
from random import Random
from lab05_monte_carlo import playout, playouts, pmcs, pmcs10_player, pmcs100_player, pmcs1000_player
from lab05_experiments import CASES
from tictactoe import empty_board

board = empty_board()
rng = Random(52000)
print("Một playout sau nước 0:", playout(board, 0, rng=rng))
samples = playouts(board, 0, N=100, rng=rng)
print(f"Sau nước 0: {samples.count(1)} thắng, {samples.count(0)} hòa, "
      f"{samples.count(-1)} thua, utility TB {sum(samples) / len(samples):.2f}")
move, details = pmcs(board, 10, rng=Random(52000), return_stats=True)
counts = {action: info["visits"] for action, info in details.items()}
print("N=10, phân bổ:", counts, "→", move)
assert sum(counts.values()) == 10 and max(counts.values()) - min(counts.values()) == 1
assert board == empty_board()
winning = CASES[0][1]
assert playouts(winning, 8, N=20, rng=Random(52000)) == [1] * 20

Một playout sau nước 0: 0
Sau nước 0: 59 thắng, 15 hòa, 26 thua, utility TB 0.33
N=10, phân bổ: {0: 1, 1: 1, 2: 2, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1} → 0


Mỗi mẫu chỉ lấy một chuỗi phản hồi. Nước 8 ở ca X thắng ngay cho utility +1 trong mọi lượt mô phỏng vì ván kết thúc ngay ở nước đó. Một nước khác cũng có thể đạt trung bình +1 khi số mẫu quá ít để bắt gặp phản hồi bất lợi.

## Độ ổn định trên các thế kiểm thử

Mỗi ngân sách chạy 30 lần trên cùng tập thế của Bài 3 và 4, với các seed từ 52000 đến 52029. Một lần chạy trước phép đo làm nóng bộ nhớ đệm kết quả bàn cờ. Các ô đồng hạng theo Minimax đều được tính là tối ưu. Ca bất lợi giả định dùng lượt X như notebook gốc để đối chiếu thuật toán. Ca thua cưỡng bức bổ sung có lượt X hợp lệ.

In [2]:
from collections import Counter
import random
from time import perf_counter

from lab05_experiments import CASES, benchmark, move_values, print_benchmark

def measure_cases(search, budgets, trials=30):
    measurements = {}
    for label, board, player in CASES:
        values = move_values(board, player)
        optimum = max(values.values())
        print(f"{label} — giá trị Minimax theo ô: {values}")
        measurements[label] = {}
        search(board, max(budgets), player, rng=random.Random(51999))
        for budget in budgets:
            generators = [random.Random(52000 + i) for i in range(trials)]
            start = perf_counter()
            choices = [search(board, budget, player, rng=rng) for rng in generators]
            ms = (perf_counter() - start) * 1000 / trials
            counts = dict(sorted(Counter(choices).items()))
            optimal = sum(count for move, count in counts.items() if values[move] == optimum)
            measurements[label][budget] = {"choices": counts, "optimal": optimal, "ms": ms}
            print(f"  N={budget:4}  ô:số lần={counts}  tối ưu={optimal}/{trials}  {ms:.4f} ms/nước")
    return measurements

trials = measure_cases(pmcs, (10, 100, 1000))

X thắng ngay — giá trị Minimax theo ô: {2: 1, 5: 0, 6: 1, 7: 0, 8: 1}
  N=  10  ô:số lần={2: 22, 5: 3, 6: 5}  tối ưu=27/30  0.0628 ms/nước
  N= 100  ô:số lần={6: 1, 8: 29}  tối ưu=30/30  0.4002 ms/nước
  N=1000  ô:số lần={8: 30}  tối ưu=30/30  3.7810 ms/nước
X phải chặn — giá trị Minimax theo ô: {3: -1, 5: -1, 6: -1, 7: 0, 8: -1}
  N=  10  ô:số lần={3: 11, 5: 4, 6: 2, 7: 11, 8: 2}  tối ưu=11/30  0.0564 ms/nước
  N= 100  ô:số lần={3: 5, 5: 3, 7: 22}  tối ưu=22/30  0.4017 ms/nước
  N=1000  ô:số lần={7: 30}  tối ưu=30/30  4.0555 ms/nước
X bất lợi (giả định) — giá trị Minimax theo ô: {1: -1, 3: -1, 4: -1, 5: -1, 6: -1, 7: -1}
  N=  10  ô:số lần={1: 6, 3: 4, 4: 12, 5: 3, 6: 3, 7: 2}  tối ưu=30/30  0.0825 ms/nước
  N= 100  ô:số lần={4: 30}  tối ưu=30/30  0.5190 ms/nước
  N=1000  ô:số lần={4: 30}  tối ưu=30/30  4.8121 ms/nước
Bàn trống — giá trị Minimax theo ô: {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0}
  N=  10  ô:số lần={0: 16, 1: 6, 2: 4, 3: 1, 4: 2, 7: 1}  tối ưu=30/30  0.6857

Ở ca X phải chặn, số lần chọn đúng ô 7 tăng từ 11/30 lên 22/30 và 30/30 khi tăng N. Mô phỏng phải lấy được các phản hồi thắng của O để nhận diện những nước X không chặn. Trên bàn trống, các nước mở đầu đều hòa khi hai bên chơi tối ưu, nên tần suất chọn một ô và tỷ lệ chọn nước tối ưu đo hai tính chất khác nhau. Ở ca thua cưỡng bức, mọi nước đều có giá trị −1. N lớn giảm nhiễu lấy mẫu và làm tăng thời gian mô phỏng.

## Đấu với Random

Mỗi cấu hình chơi 100 ván ở mỗi vai. Bộ sinh số của mô phỏng tách khỏi bộ sinh số của đối thủ. Thời gian chỉ tính lượt của tác tử PMCS, với bộ nhớ đệm kết quả bàn cờ dùng chung giữa các lượt.

In [3]:
stats = {}
for budget, agent in [(10, pmcs10_player), (100, pmcs100_player), (1000, pmcs1000_player)]:
    random.seed(53000)
    stats[budget] = benchmark(agent)
    print_benchmark(f"PMCS N={budget}", stats[budget])

print("\nN       Thắng     Hòa      Thua     ms/nước")
for budget, row in stats.items():
    total = row["games"]
    print(f"{budget:4d}   {row['wins'] / total:7.1%}  {row['draws'] / total:7.1%}  "
          f"{row['losses'] / total:7.1%}  {row['ms_per_move']:9.4f}")

PMCS N=10
  Vai X: 87 thắng, 7 hòa, 6 thua / 100 ván; 0.1188 ms/nước
  Vai O: 77 thắng, 12 hòa, 11 thua / 100 ván; 0.1064 ms/nước
  Tổng: 164 thắng, 19 hòa, 17 thua / 200 ván; 0.1129 ms/nước
PMCS N=100
  Vai X: 96 thắng, 3 hòa, 1 thua / 100 ván; 0.7794 ms/nước
  Vai O: 88 thắng, 4 hòa, 8 thua / 100 ván; 0.7865 ms/nước
  Tổng: 184 thắng, 7 hòa, 9 thua / 200 ván; 0.7828 ms/nước
PMCS N=1000
  Vai X: 100 thắng, 0 hòa, 0 thua / 100 ván; 7.9708 ms/nước
  Vai O: 93 thắng, 4 hòa, 3 thua / 100 ván; 5.7188 ms/nước
  Tổng: 193 thắng, 4 hòa, 3 thua / 200 ván; 6.8569 ms/nước

N       Thắng     Hòa      Thua     ms/nước
  10     82.0%     9.5%     8.5%     0.1129
 100     92.0%     3.5%     4.5%     0.7828
1000     96.5%     2.0%     1.5%     6.8569


Tăng N từ 100 lên 1000 nâng tỷ lệ thắng từ 92% lên 96,5% trong mẫu này và làm tăng thời gian mỗi nước. PMCS cần luật trò chơi và một chính sách mô phỏng, nên có thể áp dụng trước khi thiết kế hàm đánh giá thế cờ. Ngay cả khi có rất nhiều mẫu, ước lượng vẫn hội tụ theo kết quả trước chính sách mô phỏng ngẫu nhiên. PMCS ở N=1000 vẫn thua ba ván khi cầm O.